# Large-Scale Online Retail Mining: Market Basket Analysis & Profit Run Rate Optimization

**Business Context:**  
The Chief Merchandising Officer (`CMO`) and Chief Financial Officer (`CFO`) seek to systematically mine large-scale online retail transactions to uncover hidden item associations and translate actionable cross-selling rules into exact annual profit margin run rate increments.

**Analytical Objectives:**
1. **Programmatic Data Ingestion & Cleansing:** Ingest the UCI Online Retail dataset (`id=352`, `541,909` raw transaction lines), filter positive quantities (`Quantity > 0` and `UnitPrice > 0`), scrub credit notes (`C` prefix), and eliminate non-product overhead (`POSTAGE`, `BANK CHARGES`).
2. **Hyperparameter Optimization via FP-Growth (`mlxtend`):** Construct a Boolean one-hot transaction matrix (`19,774` invoices x `4,008` items) and run programmatic grid searches across minimum support (`min_support`) and confidence thresholds (`min_confidence`) to map rule yield and execution efficiency.
3. **Financial Conversion Model:** Mathematically translate top association rules (`Support`, `Confidence`, `Lift`) into **Annualized Profit Margin Run Rate Increments ($ / year)** assuming an `18%` checkout recommendation conversion rate across antecedent opportunity baskets ($N_{\text{opp}} = N_A - N_{A \cap B}$) at our `38%` gross contribution margin.

## 1. Environment Setup & Data Ingestion (`ucimlrepo`)

We load the official dataset (`id=352`) and apply rigorous data quality screening to isolate pure merchandise purchases.

In [1]:
import os
import time
import ssl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import fpgrowth, association_rules
from ucimlrepo import fetch_ucirepo

# Set professional plotting styling (non-glyph)
plt.style.use('default')
sns.set_theme(style="whitegrid", palette="deep")

# Load cached dataset or fetch via API
os.makedirs("data", exist_ok=True)
cache_path = os.path.join("data", "online_retail.csv")

if os.path.exists(cache_path):
    print(f"Loading cached dataset from {cache_path}...")
    df = pd.read_csv(cache_path)
else:
    print("Fetching UCI Online Retail dataset (id=352) via API...")
    ssl._create_default_https_context = ssl._create_unverified_context
    online_retail = fetch_ucirepo(id=352)
    if online_retail.data.original is not None:
        df = online_retail.data.original
    else:
        df = pd.concat([online_retail.data.features, online_retail.data.targets], axis=1)
    df.to_csv(cache_path, index=False)

# Clean dataset
clean_df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].dropna(subset=["Description"]).copy()
clean_df["Description"] = clean_df["Description"].astype(str).str.strip().str.upper()
clean_df["InvoiceNo"] = clean_df["InvoiceNo"].astype(str)
clean_df = clean_df[~clean_df["InvoiceNo"].str.upper().str.startswith("C")]

overhead_keywords = [
    "POSTAGE", "DOTCOM POSTAGE", "CARRIAGE", "CRUK COMMISSION", "BANK CHARGES",
    "DISCOUNT", "MANUAL", "AMAZON FEE", "SAMPLES", "ADJUSTMENT"
]
clean_df = clean_df[~clean_df["Description"].isin(overhead_keywords)]

print(f"Cleaned product dataset shape: {clean_df.shape[0]:,} lines across {clean_df['InvoiceNo'].nunique():,} unique invoices.")
clean_df.head()

## 2. Boolean Basket Matrix & FP-Growth Hyperparameter Optimization

We transform our clean product transactions into a one-hot encoded Boolean matrix (`basket_bool`) and execute a programmatic grid search across minimum support (`0.010` to `0.050`) and minimum confidence (`0.20` to `0.50`) to map the rule generation surface.

In [2]:
# Build one-hot Boolean basket matrix
t0 = time.time()
basket = clean_df.groupby(["InvoiceNo", "Description"])["Quantity"].sum().unstack().reset_index().fillna(0).set_index("InvoiceNo")
basket_bool = (basket > 0)
print(f"Boolean basket matrix constructed: {basket_bool.shape[0]:,} invoices x {basket_bool.shape[1]:,} distinct items in {time.time() - t0:.2f}s.")

# Run hyperparameter optimization grid search
support_grid = [0.010, 0.015, 0.020, 0.025, 0.030, 0.040, 0.050]
conf_grid = [0.20, 0.30, 0.40, 0.50]

results = []
for sup in support_grid:
    t0 = time.time()
    freq_items = fpgrowth(basket_bool, min_support=sup, use_colnames=True)
    dt = time.time() - t0
    num_itemsets = len(freq_items)
    if num_itemsets > 0:
        all_rules = association_rules(freq_items, metric="lift", min_threshold=1.1)
        for c in conf_grid:
            valid_rules = all_rules[all_rules["confidence"] >= c]
            results.append({"min_support": sup, "min_confidence": c, "num_itemsets": num_itemsets, "num_rules": len(valid_rules), "exec_time_sec": dt})
    else:
        for c in conf_grid:
            results.append({"min_support": sup, "min_confidence": c, "num_itemsets": 0, "num_rules": 0, "exec_time_sec": dt})

grid_df = pd.DataFrame(results)
pivot_grid = grid_df.pivot(index="min_support", columns="min_confidence", values="num_rules")
print("\n--- Actionable Association Rules across Hyperparameter Grid ---")
pivot_grid

### 2.1 Hyperparameter Optimization Surface Visualization

In [3]:
plt.figure(figsize=(10, 6))
for conf in sorted(grid_df["min_confidence"].unique()):
    sub = grid_df[grid_df["min_confidence"] == conf]
    plt.plot(sub["min_support"] * 100, sub["num_rules"], marker='o', linewidth=2.5, label=f"Min Confidence = {int(conf*100)}%")

plt.title("Market Basket Analysis: Hyperparameter Optimization Surface", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Minimum Support Threshold (%)", fontsize=12, fontweight='bold')
plt.ylabel("Number of Actionable Association Rules Generated", fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title="Confidence Rule", fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.savefig("hyperparameter_optimization_surface.png", dpi=300)
plt.show()

## 3. Financial Conversion Model: Association Rules to EBITDA Increments

We select the optimal regime (`min_support = 0.015`, `min_confidence = 0.30`, `lift >= 1.5`) and mathematically translate rules into annualized profit run rate increments:
$$\Delta \text{Profit}_{\text{annual}} = N_{\text{opp}} \times \alpha \times \bar{Q}_B \times P_B \times g \times \left(\frac{365.0}{T_{\text{days}}}\right)$$
where opportunity baskets $N_{\text{opp}} = N_A - N_{A \cap B}$, recommendation conversion $\alpha = 18\%$, and gross margin $g = 38\%$ across $T = 373$ days.

In [4]:
# Item financial metrics
item_metrics = clean_df.groupby("Description").agg(
    avg_price=("UnitPrice", "mean"),
    avg_qty=("Quantity", lambda x: x.sum() / x.nunique())
)

# Extract rules at optimal regime
freq_items = fpgrowth(basket_bool, min_support=0.015, use_colnames=True)
rules = association_rules(freq_items, metric="lift", min_threshold=1.5)
rules = rules[rules["confidence"] >= 0.30].copy()
rules = rules[(rules["antecedents"].apply(len) == 1) & (rules["consequents"].apply(len) == 1)].copy()
rules["antecedent"] = rules["antecedents"].apply(lambda x: list(x)[0])
rules["consequent"] = rules["consequents"].apply(lambda x: list(x)[0])

# Compute financial metrics
n_baskets = len(basket_bool)
rules["n_A"] = rules["antecedent support"] * n_baskets
rules["n_AB"] = rules["support"] * n_baskets
rules["n_opp"] = rules["n_A"] - rules["n_AB"]
rules["cons_price"] = rules["consequent"].map(item_metrics["avg_price"])
rules["cons_qty"] = rules["consequent"].map(item_metrics["avg_qty"])

days_in_dataset = (pd.to_datetime(clean_df["InvoiceDate"]).max() - pd.to_datetime(clean_df["InvoiceDate"]).min()).days
alpha = 0.18
margin = 0.38

rules["inc_rev_period"] = rules["n_opp"] * alpha * rules["cons_qty"] * rules["cons_price"]
rules["inc_profit_annual"] = rules["inc_rev_period"] * margin * (365.0 / max(days_in_dataset, 1))
rules = rules.sort_values(by="inc_profit_annual", ascending=False).reset_index(drop=True)

# Executive Table
summary_cols = ["antecedent", "consequent", "support", "confidence", "lift", "n_opp", "cons_price", "inc_profit_annual"]
summary_df = rules[summary_cols].head(10).copy()
summary_df.columns = ["Antecedent SKU", "Consequent SKU", "Support", "Conf.", "Lift", "Opp. Baskets", "Cons. Price ($)", "Annual Profit Lift ($ / yr)"]
summary_df

### 3.1 Financial Impact Visualization & Total Portfolio Run Rate

In [5]:
top_rules = rules.head(15).copy()
top_rules["rule_label"] = top_rules.apply(lambda r: f"{r['antecedent'][:22]} -> {r['consequent'][:22]} (Lift: {r['lift']:.1f}x)", axis=1)

plt.figure(figsize=(12, 8))
norm = plt.Normalize(top_rules["lift"].min(), top_rules["lift"].max())
colors = plt.cm.viridis(norm(top_rules["lift"]))

bars = plt.barh(top_rules["rule_label"][::-1], top_rules["inc_profit_annual"][::-1], color=colors[::-1], edgecolor='black', alpha=0.85)
for bar in bars:
    width = bar.get_width()
    plt.text(width + 2000, bar.get_y() + bar.get_height() / 2.0, f"${width:,.0f}/yr", va='center', ha='left', fontsize=9.5, fontweight='bold')

plt.title("Top 15 Strategic Cross-Sell Rules by Annual Profit Margin Run Rate Lift", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Annualized Incremental Gross Profit Contribution ($ / year)", fontsize=12, fontweight='bold')
plt.ylabel("Association Rule (Antecedent -> Consequent)", fontsize=12, fontweight='bold')
plt.xlim(0, top_rules["inc_profit_annual"].max() * 1.18)
plt.grid(axis='x', linestyle='--', alpha=0.6)

sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca(), pad=0.02)
cbar.set_label("Association Rule Lift (x)", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("top_association_rules_profit_impact.png", dpi=300)
plt.show()

total_portfolio_profit = rules["inc_profit_annual"].head(10).sum()
print(f"\n---> TOTAL ANNUALIZED PROFIT RUN RATE INCREMENT (Top 10 Rules Portfolio): ${total_portfolio_profit:,.2f} / year <---\n")